# Original Data From ChatGPT


In [2]:

%pip install -q ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch real patient questionnaire data from UCI
dataset = fetch_ucirepo(id=529)

X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)

print("Dataset shape:", df.shape)
print("\nFirst five records:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

df.to_csv("diabetes_uci_original.csv", index=False)
print("\nCSV saved successfully.")


Dataset shape: (520, 17)

First five records:


,age,gender,polyuria,polydipsia,sudden_weight_loss,weakness,polyphagia,genital_thrush,visual_blurring,itching,irritability,delayed_healing,partial_paresis,muscle_stiffness,alopecia,obesity,class
0,40,Male,No,Yes,No,Yes,No,No,No,Yes,No,Yes,No,Yes,Yes,Yes,Positive
1,58,Male,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,Yes,No,Positive
2,41,Male,Yes,No,No,Yes,Yes,No,No,Yes,No,Yes,No,Yes,Yes,No,Positive
3,45,Male,No,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,No,No,No,No,Positive
4,60,Male,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Positive



Column names:
['age', 'gender', 'polyuria', 'polydipsia', 'sudden_weight_loss', 'weakness', 'polyphagia', 'genital_thrush', 'visual_blurring', 'itching', 'irritability', 'delayed_healing', 'partial_paresis', 'muscle_stiffness', 'alopecia', 'obesity', 'class']

Missing values:
age                   0
gender                0
polyuria              0
polydipsia            0
sudden_weight_loss    0
weakness              0
polyphagia            0
genital_thrush        0
visual_blurring       0
itching               0
irritability          0
delayed_healing       0
partial_paresis       0
muscle_stiffness      0
alopecia              0
obesity               0
class                 0
dtype: int64

CSV saved successfully.


### Data Overview

In [3]:

# Diabetes MLOps — Step 2: Data quality checks

import pandas as pd

print("=== DATASET OVERVIEW ===")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\n=== TARGET CLASS DISTRIBUTION ===")
print(df["class"].value_counts())
print("\nPercentages:")
print(df["class"].value_counts(normalize=True) * 100)

print("\n=== DUPLICATE ANALYSIS ===")
feature_cols = [c for c in df.columns if c != "class"]

print("Exact duplicate rows:", df.duplicated().sum())
print(
    "Repeated input profiles:",
    df.duplicated(subset=feature_cols).sum()
)

conflicts = (
    df.groupby(feature_cols, dropna=False)["class"]
    .nunique()
)

print(
    "Input profiles with conflicting labels:",
    (conflicts > 1).sum()
)

print("\n=== AGE STATISTICS ===")
print(df["age"].describe())

print("\n=== UNEXPECTED TARGET VALUES ===")
print(df["class"].unique())

print("\n=== MISSING VALUES ===")
print("Total missing values:", df.isna().sum().sum())


=== DATASET OVERVIEW ===
Rows: 520
Columns: 17

=== TARGET CLASS DISTRIBUTION ===
class
Positive    320
Negative    200
Name: count, dtype: int64

Percentages:
class
Positive    61.538462
Negative    38.461538
Name: proportion, dtype: float64

=== DUPLICATE ANALYSIS ===
Exact duplicate rows: 269
Repeated input profiles: 269
Input profiles with conflicting labels: 0

=== AGE STATISTICS ===
count    520.000000
mean      48.028846
std       12.151466
min       16.000000
25%       39.000000
50%       47.500000
75%       57.000000
max       90.000000
Name: age, dtype: float64

=== UNEXPECTED TARGET VALUES ===
['Positive' 'Negative']

=== MISSING VALUES ===
Total missing values: 0


### Split and test rate

In [4]:

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Separate features and target
X = df.drop(columns=["class"])
y = df["class"].map({
    "Negative": 0,
    "Positive": 1
})

assert y.notna().all(), "Unexpected target labels found"

print("Input features:", X.shape[1])
print("Total records:", len(X))
print("\nClass distribution:")
print(y.value_counts())

# Identify repeated feature profiles
groups = pd.util.hash_pandas_object(
    X, index=False
)

print("\nExact duplicate rows:", df.duplicated().sum())
print("Repeated feature profiles:", groups.duplicated().sum())

# Keep identical feature profiles in the same split
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining positive rate:", round(y_train.mean(), 3))
print("Testing positive rate:", round(y_test.mean(), 3))

# Verify that feature profiles never cross the split
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

assert train_groups.isdisjoint(test_groups)

print("\nNo feature-profile overlap: PASSED")


Input features: 16
Total records: 520

Class distribution:
class
1    320
0    200
Name: count, dtype: int64

Exact duplicate rows: 269
Repeated feature profiles: 269

Training records: 429
Testing records: 91

Training positive rate: 0.599
Testing positive rate: 0.692

No feature-profile overlap: PASSED


### Model Training


In [5]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Identify column types
numerical_features = ["age"]
categorical_features = [
    col for col in X.columns
    if col not in numerical_features
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

# Complete ML pipeline
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train using training data only
model.fit(X_train, y_train)

print("Model training completed successfully!")


Model training completed successfully!


### Metrics

In [6]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predict only on the held-out test set
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)

print("=== MODEL EVALUATION ===")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_test,
    predictions,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))


=== MODEL EVALUATION ===
Accuracy : 0.9121
Precision: 0.9508
Recall   : 0.9206
F1 Score : 0.9355

=== CONFUSION MATRIX ===
[[25  3]
 [ 5 58]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

    Negative       0.83      0.89      0.86        28
    Positive       0.95      0.92      0.94        63

    accuracy                           0.91        91
   macro avg       0.89      0.91      0.90        91
weighted avg       0.91      0.91      0.91        91



# MLFlow Experiments

### Install and configure MLflow

In [7]:

%pip install -q mlflow

import mlflow
import mlflow.sklearn

# Use a local SQLite database for experiment tracking
mlflow.set_tracking_uri(
    "sqlite:////content/diabetes_mlflow.db"
)

mlflow.set_experiment("Diabetes-Prediction-MLOps")

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment setup successful!")


2026/11/20 05:30:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/20 05:30:20 INFO mlflow.store.db.utils: Updating database tables
2026/11/20 05:30:24 INFO mlflow.tracking.fluent: Experiment with name 'Diabetes-Prediction-MLOps' does not exist. Creating a new experiment.


MLflow version: 3.18.0
Tracking URI: sqlite:////content/diabetes_mlflow.db
Experiment setup successful!


### Record our first experiment

In [9]:

import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

mlflow.set_tracking_uri(
    "sqlite:////content/diabetes_mlflow.db"
)

mlflow.set_experiment("Diabetes-Prediction-MLOps")

# Evaluate the already-trained pipeline
predictions = model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(
        y_test, predictions, zero_division=0
    ),
    "recall": recall_score(
        y_test, predictions, zero_division=0
    ),
    "f1_score": f1_score(
        y_test, predictions, zero_division=0
    )
}

# Age is represented as float in the model example
# to avoid an unnecessarily strict integer schema.
input_example = X_test.head(3).copy()
input_example["age"] = input_example["age"].astype("float64")

with mlflow.start_run(
    run_name="Logistic-Regression-Baseline-Fixed"
) as run:

    mlflow.log_params({
        "algorithm": "LogisticRegression",
        "max_iter": 1000,
        "split_strategy": "StratifiedGroupKFold",
        "training_records": len(X_train),
        "testing_records": len(X_test)
    })

    mlflow.log_metrics(metrics)

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="diabetes_logistic_pipeline",
        serialization_format="skops",
        skops_trusted_types=[
            "sklearn.compose._column_transformer._RemainderColsList"
        ],
        input_example=input_example
    )

    mlflow.set_tag("project", "Diabetes-MLOps-Practice")
    mlflow.set_tag("dataset", "UCI-Early-Stage-Diabetes")

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)
    print("Metrics:", metrics)

print("MLflow model logging completed!")


/usr/local/lib/python3.13/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Run ID: 804bf14e86d8406c976b192135b63cee
Model URI: models:/m-22c85102835f426aac3c5429ca59f689
Metrics: {'accuracy': 0.9120879120879121, 'precision': 0.9508196721311475, 'recall': 0.9206349206349206, 'f1_score': 0.9354838709677419}
MLflow model logging completed!


### Compare the recorded results

In [10]:

# Read experiments recorded in MLflow
runs = mlflow.search_runs(
    experiment_names=["Diabetes-Prediction-MLOps"]
)

columns = [
    "tags.mlflow.runName",
    "params.algorithm",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score"
]

display(runs[[c for c in columns if c in runs.columns]])


,tags.mlflow.runName,params.algorithm,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score
0,Logistic-Regression-Baseline-Fixed,LogisticRegression,0.912088,0.95082,0.920635,0.935484
1,Logistic-Regression-Baseline,LogisticRegression,0.912088,0.95082,0.920635,0.935484


### Load the saved model back

In [11]:

import numpy as np

loaded_model = mlflow.sklearn.load_model(
    model_info.model_uri
)

original_predictions = model.predict(X_test)
reloaded_predictions = loaded_model.predict(X_test)

assert np.array_equal(
    original_predictions,
    reloaded_predictions
), "Loaded model predictions differ!"

print("Model successfully loaded from MLflow!")
print("Predictions match: PASSED")


Model successfully loaded from MLflow!
Predictions match: PASSED


## Compare models using MLflow

### Train and log both models

In [12]:

import mlflow
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

mlflow.set_tracking_uri(
    "sqlite:////content/diabetes_mlflow.db"
)
mlflow.set_experiment("Diabetes-Prediction-MLOps")

# Create a validation split from training data only
groups = pd.util.hash_pandas_object(
    X_train, index=False
)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=21
)

fit_idx, val_idx = next(
    splitter.split(X_train, y_train, groups)
)

X_fit = X_train.iloc[fit_idx]
X_val = X_train.iloc[val_idx]
y_fit = y_train.iloc[fit_idx]
y_val = y_train.iloc[val_idx]

# Confirm duplicate profiles don't cross the split
assert set(groups.iloc[fit_idx]).isdisjoint(
    set(groups.iloc[val_idx])
)

# Clone the existing pipeline configuration
logistic = clone(model)

random_forest = clone(model)
random_forest.set_params(
    classifier=RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    )
)

candidates = {
    "LogisticRegression": logistic,
    "RandomForest": random_forest
}

for name, candidate in candidates.items():

    candidate.fit(X_fit, y_fit)
    predictions = candidate.predict(X_val)

    metrics = {
        "val_accuracy": accuracy_score(y_val, predictions),
        "val_precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "val_recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "val_f1": f1_score(
            y_val, predictions, zero_division=0
        )
    }

    with mlflow.start_run(run_name=f"{name}-Validation"):
        mlflow.log_param("algorithm", name)
        mlflow.log_param("validation_strategy", "group_aware")
        mlflow.log_metrics(metrics)

    print(f"\n{name}")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}")



LogisticRegression
val_accuracy: 0.9512
val_precision: 0.9818
val_recall: 0.9474
val_f1: 0.9643

RandomForest
val_accuracy: 0.9634
val_precision: 0.9821
val_recall: 0.9649
val_f1: 0.9735


### Check consistency

In [13]:

import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score

groups = pd.util.hash_pandas_object(
    X_train, index=False
)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, estimator in {
    "LogisticRegression": logistic,
    "RandomForest": random_forest
}.items():

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_train, y_train, groups), start=1
    ):
        candidate = clone(estimator)

        candidate.fit(
            X_train.iloc[train_idx],
            y_train.iloc[train_idx]
        )

        predictions = candidate.predict(
            X_train.iloc[val_idx]
        )

        results.append({
            "model": name,
            "fold": fold,
            "f1": f1_score(
                y_train.iloc[val_idx],
                predictions,
                zero_division=0
            ),
            "recall": recall_score(
                y_train.iloc[val_idx],
                predictions,
                zero_division=0
            )
        })

results_df = pd.DataFrame(results)

print("=== RESULTS BY FOLD ===")
display(results_df)

print("\n=== AVERAGE PERFORMANCE ===")
display(
    results_df.groupby("model")[
        ["f1", "recall"]
    ].agg(["mean", "std"]).round(4)
)


=== RESULTS BY FOLD ===


,model,fold,f1,recall
0,LogisticRegression,1,0.842105,0.930233
1,LogisticRegression,2,0.916667,0.948276
2,LogisticRegression,3,0.918033,0.903226
3,LogisticRegression,4,0.961039,0.948718
4,LogisticRegression,5,0.936937,0.945455
5,RandomForest,1,0.943820,0.976744
6,RandomForest,2,0.964912,0.948276
7,RandomForest,3,0.904762,0.919355
8,RandomForest,4,0.938272,0.974359
9,RandomForest,5,1.000000,1.000000



=== AVERAGE PERFORMANCE ===


f1          recall        
                      mean     std    mean     std
model                                             
LogisticRegression  0.9150  0.0445  0.9352  0.0194
RandomForest        0.9504  0.0352  0.9637  0.0308

In [15]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=results_df)

https://docs.google.com/spreadsheets/d/13NBJn1Ak0uYez40dn-4oTUlfdVTo8dNCKsQnffrkloA/edit#gid=0


### Retrain Random Forest and register it

In [16]:

import mlflow
import mlflow.sklearn

from mlflow import MlflowClient
from sklearn.base import clone

# Use the same SQLite tracking database
TRACKING_URI = "sqlite:////content/diabetes_mlflow.db"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)
mlflow.set_experiment("Diabetes-Prediction-MLOps")

MODEL_NAME = "Diabetes-Risk-RandomForest-Educational"

# Retrain the selected model on all available training data
final_model = clone(random_forest)
final_model.fit(X_train, y_train)

# Prepare an input example for model schema inference
input_example = X_train.head(3).copy()
input_example["age"] = input_example["age"].astype("float64")

# Record and save the selected model
with mlflow.start_run(
    run_name="RandomForest-Registered-Candidate"
):

    mlflow.log_params({
        "algorithm": "RandomForest",
        "n_estimators": 100,
        "random_state": 42,
        "selection_method": "5-fold-group-aware-CV"
    })

    # Log the results we already measured
    rf_results = results_df[
        results_df["model"] == "RandomForest"
    ]

    mlflow.log_metrics({
        "cv_mean_f1": float(rf_results["f1"].mean()),
        "cv_mean_recall": float(rf_results["recall"].mean())
    })

    # Keep skops serialization and explicitly approve
    # only these known sklearn internal types
    logged_model = mlflow.sklearn.log_model(
        sk_model=final_model,
        name="random_forest_pipeline",
        serialization_format="skops",
        skops_trusted_types=[
            "sklearn.compose._column_transformer._RemainderColsList",
            "sklearn.tree._tree.Tree"
        ],
        input_example=input_example
    )

# Register the saved MLflow model
registered = mlflow.register_model(
    model_uri=logged_model.model_uri,
    name=MODEL_NAME
)

print("Registered model:", registered.name)
print("Model version:", registered.version)
print("Source URI:", logged_model.model_uri)


/usr/local/lib/python3.13/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model: Diabetes-Risk-RandomForest-Educational
Model version: 1
Source URI: models:/m-c957b2067c104177b9a430711787938a


Successfully registered model 'Diabetes-Risk-RandomForest-Educational'.
Created version '1' of model 'Diabetes-Risk-RandomForest-Educational'.


### Assign an alias and load from the registry

In [17]:

import numpy as np

client = MlflowClient()

# Mark the registered version as a candidate
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="candidate",
    version=registered.version
)

# Load using the registry alias
registry_uri = f"models:/{MODEL_NAME}@candidate"

registered_model = mlflow.sklearn.load_model(
    registry_uri
)

# Compare predictions with the trained model
expected = final_model.predict(X_train.head(10))
actual = registered_model.predict(X_train.head(10))

np.testing.assert_array_equal(expected, actual)

print("Registry model loaded: PASSED")
print("Registered version:", registered.version)
print("Prediction consistency: PASSED")
print("Alias: candidate")


Registry model loaded: PASSED
Registered version: 1
Prediction consistency: PASSED
Alias: candidate


# BentoML Model Serving

### Install BentoML in Google Colab

In [18]:

%pip install -q bentoml

import bentoml
import mlflow

print("BentoML version:", bentoml.__version__)
print("MLflow version:", mlflow.__version__)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
BentoML version: 1.4.40
MLflow version: 3.18.0


### Import the registered MLflow model

In [19]:

import bentoml
import mlflow

mlflow.set_tracking_uri(
    "sqlite:////content/diabetes_mlflow.db"
)
mlflow.set_registry_uri(
    "sqlite:////content/diabetes_mlflow.db"
)

MODEL_NAME = "Diabetes-Risk-RandomForest-Educational"

# Fetch the model we registered previously
model_uri = f"models:/{MODEL_NAME}@candidate"

# Import into the BentoML Model Store
bento_model = bentoml.mlflow.import_model(
    "diabetes_risk_educational",
    model_uri
)

print("BentoML model imported!")
print("Model tag:", bento_model.tag)

print("\nAvailable BentoML models:")
for item in bentoml.models.list():
    print(item.tag)


BentoML model imported!
Model tag: diabetes_risk_educational:5kx4n5xeu6anqasc

Available BentoML models:
diabetes_risk_educational:5kx4n5xeu6anqasc


### Test the imported model

In [21]:

import bentoml
import numpy as np
import pandas as pd

# Load model from BentoML
bento_loaded = bentoml.mlflow.load_model(
    str(bento_model.tag)
)

# Prepare test input
sample = X_test.head(3).copy()

# Fix schema mismatch
sample["age"] = sample["age"].astype("float64")

print("=== INPUT DATA TYPES ===")
print(sample.dtypes)

# Compare original model with BentoML model
expected = final_model.predict(sample)
actual = bento_loaded.predict(sample)

print("\nOriginal predictions:", expected)
print("BentoML predictions:", actual)

np.testing.assert_array_equal(expected, actual)

print("\nBentoML model loading: PASSED")
print("Prediction consistency: PASSED")


=== INPUT DATA TYPES ===
age                   float64
gender                 object
polyuria               object
polydipsia             object
sudden_weight_loss     object
weakness               object
polyphagia             object
genital_thrush         object
visual_blurring        object
itching                object
irritability           object
delayed_healing        object
partial_paresis        object
muscle_stiffness       object
alopecia               object
obesity                object
dtype: object

Original predictions: [1 0 1]
BentoML predictions: [1 0 1]

BentoML model loading: PASSED
Prediction consistency: PASSED


# Create service.py in Colab

In [22]:

%%writefile /content/service.py
import bentoml
import pandas as pd

from typing import Literal
from pydantic import BaseModel, Field, ConfigDict

YesNo = Literal["Yes", "No"]

class PatientInput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    age: float = Field(ge=1, le=120)
    gender: Literal["Male", "Female"]
    polyuria: YesNo
    polydipsia: YesNo
    sudden_weight_loss: YesNo
    weakness: YesNo
    polyphagia: YesNo
    genital_thrush: YesNo
    visual_blurring: YesNo
    itching: YesNo
    irritability: YesNo
    delayed_healing: YesNo
    partial_paresis: YesNo
    muscle_stiffness: YesNo
    alopecia: YesNo
    obesity: YesNo


@bentoml.service
class DiabetesClassifier:

    model_ref = bentoml.models.BentoModel(
        "diabetes_risk_educational:latest"
    )

    def __init__(self):
        self.model = bentoml.mlflow.load_model(
            self.model_ref
        )

    @bentoml.api
    def predict(self, input_data: PatientInput) -> dict:

        # Convert validated request into model input
        record = input_data.model_dump()
        frame = pd.DataFrame([record])

        # Ensure expected numeric type
        frame["age"] = frame["age"].astype("float64")

        prediction = int(
            self.model.predict(frame)[0]
        )

        return {
            "prediction": prediction,
            "label": (
                "Positive" if prediction == 1
                else "Negative"
            ),
            "model": "RandomForest-Educational"
        }


Writing /content/service.py


## Start the BentoML server

In [23]:

import subprocess
import time
import requests

log_file = open("/content/bentoml_server.log", "w")

server = subprocess.Popen(
    [
        "bentoml", "serve",
        "service:DiabetesClassifier",
        "--host", "127.0.0.1",
        "--port", "3000"
    ],
    cwd="/content",
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Check whether the API becomes ready
ready = False

for attempt in range(45):
    try:
        response = requests.get(
            "http://127.0.0.1:3000/readyz",
            timeout=2
        )

        if response.status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass

    if server.poll() is not None:
        break

    time.sleep(1)

if ready:
    print("BentoML server: READY")
    print("API: http://127.0.0.1:3000/predict")
else:
    print("Server not ready. Recent logs:")
    print(open("/content/bentoml_server.log").read()[-4000:])


BentoML server: READY
API: http://127.0.0.1:3000/predict


### Send your first HTTP prediction request

In [24]:

import requests

# Use an existing example from our test dataset
sample = X_test.iloc[0].to_dict()

# Convert age to match the MLflow schema
sample["age"] = float(sample["age"])

response = requests.post(
    "http://127.0.0.1:3000/predict",
    json={"input_data": sample},
    timeout=30
)

print("HTTP status:", response.status_code)
print("API response:")
print(response.text)

response.raise_for_status()


HTTP status: 200
API response:
{"prediction": 1, "label": "Positive", "model": "RandomForest-Educational"}


# Preserve your model before Docker deployment

### Export the model and prepare the files

In [25]:

import bentoml
import sys
import json
import shutil
from pathlib import Path
from importlib.metadata import version

project = Path("/content/diabetes-deployment")
project.mkdir(exist_ok=True)

# Export the exact model version we tested
model_tag = str(bento_model.tag)

exported = bentoml.models.export_model(
    model_tag,
    str(project / "diabetes_model.bentomodel")
)

# Save the service with its exact model version
source = Path("/content/service.py").read_text()

assert "diabetes_risk_educational:latest" in source

source = source.replace(
    "diabetes_risk_educational:latest",
    model_tag
)

(project / "service.py").write_text(source)

# Record the exact dependency versions
packages = [
    "bentoml",
    "mlflow",
    "scikit-learn",
    "pandas",
    "numpy",
    "pydantic",
    "skops"
]

requirements = "\n".join(
    f"{pkg}=={version(pkg)}"
    for pkg in packages
)

(project / "requirements.txt").write_text(
    requirements + "\n"
)

# Save a known example for later API testing
test_sample = X_test.iloc[0].to_dict()
test_sample["age"] = float(test_sample["age"])

(project / "test_request.json").write_text(
    json.dumps(
        {"input_data": test_sample},
        indent=2
    )
)

(project / "environment.txt").write_text(
    f"Python: {sys.version}\n"
    f"BentoML model: {model_tag}\n"
)

print("Model exported:", exported)
print("Python:", sys.version.split()[0])
print("Model tag:", model_tag)
print("Files prepared:")
for file in project.iterdir():
    print(" -", file.name)


Model exported: /content/diabetes-deployment/diabetes_model.bentomodel
Python: 3.13.18
Model tag: diabetes_risk_educational:5kx4n5xeu6anqasc
Files prepared:
 - test_request.json
 - environment.txt
 - service.py
 - diabetes_model.bentomodel
 - requirements.txt


### Download the complete deployment package

In [26]:

import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/diabetes-deployment",
    "zip",
    root_dir="/content/diabetes-deployment"
)

print("Deployment package:", archive)
files.download(archive)


Deployment package: /content/diabetes-deployment.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>